<a href="https://colab.research.google.com/github/mahmudscode/NLP_work/blob/main/NLP_WORK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Define the target path
target_path = '/content/drive/MyDrive/LLM_BANGLA_DATA/Bangla_newspaper'

# Change the current working directory
os.chdir(target_path)

# Verify the current working directory
print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/LLM_BANGLA_DATA/Bangla_newspaper


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout,
    Bidirectional,
    LSTM
)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [4]:
import os

for file in os.listdir():
    print(file)

data
data_v2


In [5]:
import os

for root, dirs, files in os.walk("data_v2"):
    print(f"\nFolder: {root}")
    print("Files:", files[:5])
    break


Folder: data_v2
Files: ['data_v2.json']


In [6]:
import os

for root, dirs, files in os.walk("data_v2"):
    if len(files) > 0:
        print("Example file:", os.path.join(root, files[0]))
        break

Example file: data_v2/data_v2.json


In [7]:
import json

with open("data_v2/data_v2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))

<class 'list'>


In [8]:
print(len(data))

408471


In [9]:
data[0]

{'author': 'গাজীপুর প্রতিনিধি',
 'category': 'bangladesh',
 'category_bn': 'বাংলাদেশ',
 'published_date': '০৪ জুলাই ২০১৩, ২৩:২৬',
 'modification_date': '০৪ জুলাই ২০১৩, ২৩:২৭',
 'tag': ['গাজীপুর'],
 'comment_count': 0,
 'title': 'কালিয়াকৈরে টিফিন খেয়ে ৫০০ শ্রমিক অসুস্থ, বিক্ষোভ',
 'url': 'http://www.prothom-alo.com/bangladesh/article/19030',
 'content': 'গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায় আজ বৃহস্পতিবার রাতের টিফিন খেয়ে একটি পোশাক কারখানার ৫০০ শ্রমিক অসুস্থ হয়ে পড়েছেন। এ ঘটনায় বিক্ষোভ করেছেন ওই কারখানার শ্রমিকেরা।সফিপুর মডার্ন হাসপাতালের জরুরি বিভাগের চিকিত্সক আল আমিন প্রথম আলো ডটকমকে বলেন, খাদ্যে বিষক্রিয়ায় তাঁরা (শ্রমিকেরা) অসুস্থ হয়ে পড়েছেন। এতে আতঙ্কিত হওয়ার কিছু নেই। অসুস্থদের চিকিত্সা দেওয়া হয়েছে।কারখানার শ্রমিক ও পুলিশ সূত্রে জানা যায়, উপজেলার তেলিরচালা এলাকার সেজাদ সোয়েটার লিমিটেড কারখানার শ্রমিকদের আজ রাত সাড়ে সাতটার দিকে টিফিন দেওয়া হয়। টিফিনে ছিল ডিম, রুটি, পেটিস ও কলা। টিফিন খেয়ে শ্রমিকেরা যথারীতি কাজে যোগ দেন। ওই টিফিন খাওয়ার প্রায় এক ঘণ্টা 

In [10]:
import json
import pandas as pd

with open("data_v2/data_v2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.shape)

df.head()

(408471, 10)


,author,category,category_bn,published_date,modification_date,tag,comment_count,title,url,content
0,গাজীপুর প্রতিনিধি,bangladesh,বাংলাদেশ,"০৪ জুলাই ২০১৩, ২৩:২৬","০৪ জুলাই ২০১৩, ২৩:২৭",[গাজীপুর],0,"কালিয়াকৈরে টিফিন খেয়ে ৫০০ শ্রমিক অসুস্থ, বিক...",http://www.prothom-alo.com/bangladesh/article/...,গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায়...
1,অনলাইন ডেস্ক,sports,খেলা,"০৪ জুলাই ২০১৩, ২৩:০৯","০৪ জুলাই ২০১৩, ২৩:১১",[টেনিস],0,সেমিফাইনাল বাধাও পেরিয়ে গেলেন লিসিকি,http://www.prothom-alo.com/sports/article/19028,এবারের উইম্বলডনটা স্মরণীয় করে রাখার মিশনেই যে...
2,নিজস্ব প্রতিবেদক,bangladesh,বাংলাদেশ,"০৪ জুলাই ২০১৩, ২২:২৫","০৪ জুলাই ২০১৩, ২৩:১২",[রাজনীতি],0,সংসদে খালেদার অভিযোগের জবাব দিয়েছে ভারত,http://www.prothom-alo.com/bangladesh/article/...,জাতীয় সংসদে বিএনপি চেয়ারপারসন ও বিরোধীদলীয় ...
3,অনলাইন ডেস্ক,technology,বিজ্ঞান ও প্রযুক্তি,"০৪ জুলাই ২০১৩, ২১:৩৭","০৪ জুলাই ২০১৩, ২১:৪৫",[গবেষণা],0,পাসওয়ার্ড ভুলে যান!,http://www.prothom-alo.com/technology/article/...,সহজ পাসওয়ার্ডের কারণে অনলাইন অ্যাকাউন্ট সহজেই...
4,অনলাইন ডেস্ক,technology,বিজ্ঞান ও প্রযুক্তি,"০৪ জুলাই ২০১৩, ২১:৩৫","০৪ জুলাই ২০১৩, ২১:৩৭",[কম্পিউটার],0,চলে গেলেন মাউস উদ্ভাবক,http://www.prothom-alo.com/technology/article/...,কম্পিউটার মাউসের উদ্ভাবক ডগলাস অ্যাঙ্গেলবার্ট ...


In [11]:
df = df[
    [
        "title",
        "content",
        "category"
    ]
]

df.head()

,title,content,category
0,"কালিয়াকৈরে টিফিন খেয়ে ৫০০ শ্রমিক অসুস্থ, বিক...",গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায়...,bangladesh
1,সেমিফাইনাল বাধাও পেরিয়ে গেলেন লিসিকি,এবারের উইম্বলডনটা স্মরণীয় করে রাখার মিশনেই যে...,sports
2,সংসদে খালেদার অভিযোগের জবাব দিয়েছে ভারত,জাতীয় সংসদে বিএনপি চেয়ারপারসন ও বিরোধীদলীয় ...,bangladesh
3,পাসওয়ার্ড ভুলে যান!,সহজ পাসওয়ার্ডের কারণে অনলাইন অ্যাকাউন্ট সহজেই...,technology
4,চলে গেলেন মাউস উদ্ভাবক,কম্পিউটার মাউসের উদ্ভাবক ডগলাস অ্যাঙ্গেলবার্ট ...,technology


In [12]:
df["text"] = (
    df["title"].fillna("")
    + " "
    + df["content"].fillna("")
)

df = df[
    [
        "text",
        "category"
    ]
]

df.head()

,text,category
0,"কালিয়াকৈরে টিফিন খেয়ে ৫০০ শ্রমিক অসুস্থ, বিক...",bangladesh
1,সেমিফাইনাল বাধাও পেরিয়ে গেলেন লিসিকি এবারের উ...,sports
2,সংসদে খালেদার অভিযোগের জবাব দিয়েছে ভারত জাতীয...,bangladesh
3,পাসওয়ার্ড ভুলে যান! সহজ পাসওয়ার্ডের কারণে অন...,technology
4,চলে গেলেন মাউস উদ্ভাবক কম্পিউটার মাউসের উদ্ভাব...,technology


In [13]:
df = df.dropna()

print(df.shape)

(408471, 2)


In [14]:
df = df.drop_duplicates(
    subset=["text"]
)

print(df.shape)

(407078, 2)


In [15]:
df["category"].value_counts()

,count
category,
bangladesh,231859
sports,48987
international,30835
entertainment,29866
economy,17224
opinion,15694
technology,12107
life-style,10842
education,9664


In [16]:
import re

def clean_text(text):

    text = str(text)

    text = re.sub(
        r"http\S+",
        "",
        text
    )

    text = re.sub(
        r"www\S+",
        "",
        text
    )

    text = re.sub(
        r"\n",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [17]:
df["clean_text"] = df["text"].apply(
    clean_text
)

In [18]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["label"] = encoder.fit_transform(
    df["category"]
)

label_map = dict(
    zip(
        encoder.classes_,
        encoder.transform(
            encoder.classes_
        )
    )
)

print(label_map)

{'bangladesh': np.int64(0), 'economy': np.int64(1), 'education': np.int64(2), 'entertainment': np.int64(3), 'international': np.int64(4), 'life-style': np.int64(5), 'opinion': np.int64(6), 'sports': np.int64(7), 'technology': np.int64(8)}


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

MAX_WORDS = 100000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(
    X_train
)

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(
    X_train
)

X_test_seq = tokenizer.texts_to_sequences(
    X_test
)

In [ ]:
lengths = [len(x) for x in X_train_seq]

pd.Series(lengths).describe()

In [ ]:
pd.Series(lengths).quantile(
    [0.90,0.95,0.99]
)